# Light Jet Market Analysis

In [2]:
# ============================================================
# CELL 1: SETUP & LOAD - Light Jet inter-metro audit
# ============================================================
import pandas as pd
import numpy as np

METRO_PATH = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"
print("Loading and processing Light Jet market audit...")
df = pd.read_parquet(METRO_PATH)

df['FlightDate_ET'] = pd.to_datetime(df['FlightDate_ET'])
df = df.rename(columns={'FromCluster': 'FromMetro', 'ToCluster': 'ToMetro'})

lj_models = ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra']

mask = (
    (df['aircraft_segment'] == 'Light Jet') &
    (df['aircraft_model'].isin(lj_models)) &
    (df['FromMetro'] != 'OTHER_METRO') &
    (df['ToMetro'] != 'OTHER_METRO') &
    (df['FromMetro'] != df['ToMetro']) &
    (df['Hours'] >= 1.5)
)
lj_lh = df[mask].copy()
lj_lh['corridor'] = lj_lh['FromMetro'] + " -> " + lj_lh['ToMetro']

wup_mask = ((lj_lh['Operator'].str.contains('Wheels Up', case=False, na=False)) |
            (lj_lh['Operator'] == 'Mountain Aviation'))
lj_lh['is_wup'] = wup_mask

def make_pair_key(row):
    return " <-> ".join(sorted([row['FromMetro'], row['ToMetro']]))
lj_lh['pair_key'] = lj_lh.apply(make_pair_key, axis=1)

pair_analysis = lj_lh.groupby('pair_key').agg(
    total_pair_missions=('pair_key', 'size'),
    total_wup_missions=('is_wup', 'sum'),
    avg_hours_combined=('Hours', 'mean')
).reset_index()

def get_splits(group):
    counts = group['corridor'].value_counts()
    hrs = group.groupby('corridor')['Hours'].mean()
    wup = group.groupby('corridor')['is_wup'].sum()
    total = group.groupby('corridor').size()
    return pd.Series({
        'mission_split': " | ".join(counts.astype(str)),
        'hours_split': " | ".join([f"{h:.2f}" for h in hrs]),
        'share_split': " | ".join([f"{(w/t)*100:.1f}%" for w, t in zip(wup, total)])
    })

splits = lj_lh.groupby('pair_key').apply(get_splits, include_groups=False).reset_index()
pair_analysis = pair_analysis.merge(splits, on='pair_key').sort_values('total_pair_missions', ascending=False)

intra_metro_count = lj_lh[lj_lh['FromMetro'] == lj_lh['ToMetro']].shape[0]
print(f"LJ Market Audit Complete.")
print(f"1. Total Inter-Metro Missions: {len(lj_lh):,}")
print(f"2. Unique Directional Corridors: {lj_lh['corridor'].nunique()}")
print(f"3. Intra-Metro (Same City) Missions: {intra_metro_count}")
display(pair_analysis.head(10))

Loading and processing Light Jet market audit...
LJ Market Audit Complete.
1. Total Inter-Metro Missions: 102,664
2. Unique Directional Corridors: 430
3. Intra-Metro (Same City) Missions: 0


,pair_key,total_pair_missions,total_wup_missions,avg_hours_combined,mission_split,hours_split,share_split
124,Denver <-> LA Basin,4435,90,1.910210,2475 | 1960,1.96 | 1.84,1.8% | 2.3%
196,New York City <-> South Florida,3718,89,2.759651,1869 | 1849,2.84 | 2.68,2.3% | 2.5%
107,Dallas <-> Denver,3095,71,1.989965,1831 | 1264,2.06 | 1.88,2.3% | 2.2%
25,Bay Area <-> Denver,2601,48,2.007510,1463 | 1138,1.89 | 2.10,2.3% | 1.5%
89,Chicago <-> South Florida,2434,51,2.701342,1229 | 1205,2.61 | 2.79,1.8% | 2.4%
105,DMV <-> South Florida,2240,88,2.227723,1125 | 1115,2.30 | 2.15,4.2% | 3.6%
168,LA Basin <-> Seattle,2122,21,2.143583,1092 | 1030,2.15 | 2.14,0.9% | 1.1%
72,Charlotte <-> South Florida,2078,94,1.722891,1067 | 1011,1.74 | 1.70,4.8% | 4.3%
82,Chicago <-> New York City,2059,49,1.860733,1098 | 961,1.73 | 1.97,2.7% | 2.1%
19,Atlanta <-> South Florida,2001,72,1.657854,1195 | 806,1.64 | 1.67,4.2% | 3.2%


In [3]:
# ============================================================
# CELL 2: DIRECTIONAL SEASONAL DISCOVERY - KMEANS
# ============================================================
from sklearn.cluster import KMeans

seasonal_counts = (lj_lh.groupby(["corridor", "month"]).size().unstack(fill_value=0))
seasonal_counts = seasonal_counts.reindex(columns=range(1, 13), fill_value=0)

MIN_MISSIONS = 20
annual_missions = seasonal_counts.sum(axis=1)
relevant_corridors = annual_missions > MIN_MISSIONS
seasonal_counts_relevant = seasonal_counts.loc[relevant_corridors].copy()

monthly_density = seasonal_counts_relevant.div(seasonal_counts_relevant.sum(axis=1), axis=0)
seasonality_index = monthly_density / (1 / 12)

km = KMeans(n_clusters=4, random_state=42, n_init=50)
cluster_ids = km.fit_predict(monthly_density)

X_discovery = monthly_density.copy()
X_discovery["cluster_id"] = cluster_ids
X_discovery["annual_missions"] = annual_missions.loc[relevant_corridors] / 2

for m in range(1, 13):
    X_discovery[f"seasonality_index_m{m}"] = seasonality_index[m]

cluster_centers_density = (X_discovery.groupby("cluster_id")[list(range(1, 13))].mean().reindex(columns=range(1, 13)))
print(f"Processed {len(X_discovery)} LJ corridors into 4 seasonal demand regimes.")
print("\nMONTHLY DENSITY BY CLUSTER:")
display(cluster_centers_density.style.background_gradient(axis=1, cmap="YlOrRd").format("{:.1%}"))

seasonality_cols = [f"seasonality_index_m{m}" for m in range(1, 13)]
cluster_centers_index = (X_discovery.groupby("cluster_id")[seasonality_cols].mean())
cluster_centers_index.columns = range(1, 13)
print("\nSEASONALITY INDEX BY CLUSTER:")
display(cluster_centers_index.style.background_gradient(axis=1, cmap="YlOrRd").format("{:.2f}x"))

Processed 360 LJ corridors into 4 seasonal demand regimes.

MONTHLY DENSITY BY CLUSTER:


month,1,2,3,4,5,6,7,8,9,10,11,12
cluster_id,,,,,,,,,,,,
0,11.0%,10.4%,10.8%,10.2%,9.1%,5.2%,4.9%,4.3%,5.4%,7.7%,10.3%,10.6%
1,5.5%,5.2%,5.0%,5.8%,8.9%,10.2%,16.3%,11.4%,11.9%,8.2%,6.1%,5.6%
2,4.3%,4.7%,5.7%,7.5%,8.9%,11.2%,7.9%,10.6%,11.8%,12.5%,9.6%,5.3%
3,7.2%,7.4%,8.1%,9.2%,9.3%,7.9%,7.6%,8.2%,8.3%,10.1%,8.4%,8.5%



SEASONALITY INDEX BY CLUSTER:


,1,2,3,4,5,6,7,8,9,10,11,12
cluster_id,,,,,,,,,,,,
0,1.32x,1.25x,1.29x,1.22x,1.09x,0.63x,0.59x,0.52x,0.65x,0.92x,1.24x,1.28x
1,0.66x,0.62x,0.59x,0.69x,1.07x,1.23x,1.95x,1.37x,1.42x,0.99x,0.73x,0.67x
2,0.52x,0.56x,0.68x,0.90x,1.07x,1.34x,0.95x,1.27x,1.42x,1.51x,1.15x,0.64x
3,0.86x,0.89x,0.97x,1.10x,1.11x,0.95x,0.91x,0.98x,1.00x,1.21x,1.01x,1.02x


In [4]:
# ============================================================
# CELL 3: LABELED SEASONALITY AUDIT - 4 archetypes
# ============================================================
final_labels_lj = {
    0: "winter-spring-peak",
    1: "summer-peak",
    2: "core-utilization",
    3: "multi-peak-directional"
}

labeled_centers = X_discovery.groupby("cluster_id")[list(range(1, 13))].mean()
labeled_centers.index = [f"{i}: {final_labels_lj.get(i, 'Unknown')}" for i in labeled_centers.index]

print("FINAL LJ ARCHETYPE HEATMAP (Monthly Density %):")
display(labeled_centers.style.background_gradient(axis=1, cmap="YlOrRd").format("{:.1%}"))

summer_jump = labeled_centers[6] - labeled_centers[5]
top_toggle = summer_jump.idxmax()
print(f"\nStrongest June Toggle candidate: {top_toggle}")
print(f"Momentum: +{summer_jump.max():.1%} absolute density increase MoM.")

FINAL LJ ARCHETYPE HEATMAP (Monthly Density %):


month,1,2,3,4,5,6,7,8,9,10,11,12
0: winter-spring-peak,11.0%,10.4%,10.8%,10.2%,9.1%,5.2%,4.9%,4.3%,5.4%,7.7%,10.3%,10.6%
1: summer-peak,5.5%,5.2%,5.0%,5.8%,8.9%,10.2%,16.3%,11.4%,11.9%,8.2%,6.1%,5.6%
2: core-utilization,4.3%,4.7%,5.7%,7.5%,8.9%,11.2%,7.9%,10.6%,11.8%,12.5%,9.6%,5.3%
3: multi-peak-directional,7.2%,7.4%,8.1%,9.2%,9.3%,7.9%,7.6%,8.2%,8.3%,10.1%,8.4%,8.5%



Strongest June Toggle candidate: 2: core-utilization
Momentum: +2.3% absolute density increase MoM.


In [5]:
# ============================================================
# CELL 4: FINAL VALIDATION - KMeans universe summary
# Expected: 360 corridors, 51,049 annualized flights
# ============================================================
total_corridors = len(X_discovery)
total_flights = X_discovery['annual_missions'].sum()

stats = X_discovery.groupby('cluster_id').agg({'annual_missions': ['count', 'sum']})
stats.index = stats.index.map(final_labels_lj)
stats.columns = ['corridor_count', 'flight_volume']
stats['volume_pct'] = (stats['flight_volume'] / total_flights) * 100

intra_metro_issues = [c for c in X_discovery.index if c.split(' -> ')[0] == c.split(' -> ')[1]]

print("--- LIGHT JET CLUSTER UNIVERSE SUMMARY ---")
print(f"Filtering Criteria Applied:")
print(f"   - Segment      : Light Jet (Select Models)")
print(f"   - Min Leg Time : >= 1.5 Hours")
print(f"   - Relevance    : > {MIN_MISSIONS} missions total (2-yr)")
print(f"   - Logic        : Inter-Metro Only (No Same-City)")
print(f"\nTotal Inter-Metro Corridors : {total_corridors}")
print(f"Total Annual Flight Volume  : {total_flights:,.0f} (Annualized /2)")
print("\nLJ ARCHETYPE BREAKDOWN (Volume & Count):")
print(f"{'Archetype':<25} | {'Corridors':<10} | {'Flights':<12} | {'Volume %'}")
print("-" * 65)
for label, row in stats.sort_values('flight_volume', ascending=False).iterrows():
    print(f"{label:<25} | {int(row['corridor_count']):<10} | {int(row['flight_volume']):<12,} | {row['volume_pct']:.1f}%")
print("\n--- SAFETY CHECKS ---")
if not intra_metro_issues:
    print(f"Final Validation Passed: 0 intra-metro corridors found.")
else:
    print(f"Final Validation FAILED: Found {len(intra_metro_issues)} intra-metro records!")

--- LIGHT JET CLUSTER UNIVERSE SUMMARY ---
Filtering Criteria Applied:
   - Segment      : Light Jet (Select Models)
   - Min Leg Time : >= 1.5 Hours
   - Relevance    : > 20 missions total (2-yr)
   - Logic        : Inter-Metro Only (No Same-City)

Total Inter-Metro Corridors : 360
Total Annual Flight Volume  : 51,049 (Annualized /2)

LJ ARCHETYPE BREAKDOWN (Volume & Count):
Archetype                 | Corridors  | Flights      | Volume %
-----------------------------------------------------------------
multi-peak-directional    | 147        | 22,416       | 43.9%
winter-spring-peak        | 88         | 17,122       | 33.5%
core-utilization          | 76         | 6,399        | 12.5%
summer-peak               | 49         | 5,111        | 10.0%

--- SAFETY CHECKS ---
Final Validation Passed: 0 intra-metro corridors found.


In [6]:
# ============================================================
# CELL 5: COMPREHENSIVE DATASET EXPORT - lj_comprehensive.csv
# Output: 4,320 rows (360 corridors x 12 months)
# ============================================================

OUT_PATH = "lj_comprehensive.csv"

comp_base = X_discovery[list(range(1, 13))].copy()
comp_base['cluster_label'] = comp_base.index.map(lambda c: final_labels_lj[X_discovery.loc[c, 'cluster_id']])
comp_base['corridor'] = comp_base.index
comp_long = pd.melt(comp_base, id_vars=['cluster_label', 'corridor'], var_name='month', value_name='density_pct')
comp_long['seasonality_index'] = (comp_long['density_pct'] * 12 / 100).round(4)

bins = [0, 4, 6, 9, 12, 15, float("inf")]
labels = ["< 4%", "4% - 6%", "6% - 9%", "9% - 12%", "12% - 15%", "> 15%"]
comp_long['slab'] = pd.cut(comp_long['density_pct'] * 100, bins=bins, labels=labels)

comp_long['intensity'] = pd.cut(comp_long['density_pct'] * 100, bins=bins, labels=False)

comp_long['month'] = comp_long['month'].map({
    1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
    7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'
})

comp_out = comp_long[['cluster_label', 'corridor', 'month', 'density_pct', 'seasonality_index', 'slab', 'intensity']]
comp_out.to_csv(OUT_PATH, index=False)
print(f"Exported: {OUT_PATH}")
print(f"Shape: {comp_out.shape}")
print(f"Expected: (4320, 7) for 360 corridors x 12 months")
display(comp_out.head(10))

Exported: lj_comprehensive.csv
Shape: (4320, 7)
Expected: (4320, 7) for 360 corridors x 12 months


,cluster_label,corridor,month,density_pct,seasonality_index,slab,intensity
0,core-utilization,Atlanta -> Boston,Jan,0.038462,0.0046,< 4%,0.0
1,multi-peak-directional,Atlanta -> Chicago,Jan,0.079208,0.0095,6% - 9%,2.0
2,multi-peak-directional,Atlanta -> DMV,Jan,0.062500,0.0075,6% - 9%,2.0
3,winter-spring-peak,Atlanta -> Dallas,Jan,0.093431,0.0112,9% - 12%,3.0
4,multi-peak-directional,Atlanta -> Denver,Jan,0.073529,0.0088,6% - 9%,2.0
5,summer-peak,Atlanta -> Detroit,Jan,0.043011,0.0052,4% - 6%,1.0
6,winter-spring-peak,Atlanta -> Houston,Jan,0.101227,0.0121,9% - 12%,3.0
7,multi-peak-directional,Atlanta -> Minneapolis,Jan,0.065891,0.0079,6% - 9%,2.0
8,multi-peak-directional,Atlanta -> New York City,Jan,0.051988,0.0062,4% - 6%,1.0
9,multi-peak-directional,Atlanta -> North Florida,Jan,0.046512,0.0056,4% - 6%,1.0


---
# MAY SEASONALITY ANALYSIS
## Quick analysis for May across key corridors

In [8]:


# MAY SEASONALITY ANALYSIS
# Target corridors: Denver, Charlotte, Chicago, NYC, South Florida, Phoenix, LA Basin
# ============================================================================

TARGET_CORRIDORS = [
    'Denver -> LA Basin', 'Charlotte -> South Florida', 'Chicago -> South Florida',
    'LA Basin -> Denver', 'South Florida -> Charlotte', 'South Florida -> Chicago',
    'New York City -> South Florida', 'South Florida -> New York City',
    'Denver -> South Florida', 'South Florida -> Denver',
    'Boston -> Phoenix Valley', 'New York City -> LA Basin',
]

# Filter to May
may_data = comp_out[(comp_out['corridor'].isin(TARGET_CORRIDORS)) & (comp_out['month'] == 'May')].copy()

# Compute si_multiplier
def si_to_multiplier(si):
    if si < 0.75:   return 0.75
    elif si < 0.90: return 0.90
    elif si < 1.10: return 1.00
    elif si < 1.30: return 1.15
    elif si < 1.60: return 1.30
    else:           return 1.50

may_data['si_multiplier'] = may_data['seasonality_index'].apply(si_to_multiplier)

# Summary table sorted by SI
may_summary = may_data[['corridor', 'cluster_label', 'density_pct', 'seasonality_index', 'si_multiplier']].sort_values('seasonality_index', ascending=False)

print("=" * 100)
print("MAY SEASONALITY - TARGET CORRIDORS")
print("=" * 100)
print()

for idx, row in may_summary.iterrows():
    print(f"{row['corridor']:<35} | {row['cluster_label']:<22} | SI: {row['seasonality_index']:>5.2f}x | Mult: {row['si_multiplier']:>5.2f}")

print()
print("=" * 100)
print("MAY SUMMARY STATISTICS")
print("=" * 100)
print(f"Average SI: {may_data['seasonality_index'].mean():.2f}x")
print(f"SI Range: {may_data['seasonality_index'].min():.2f}x to {may_data['seasonality_index'].max():.2f}x")
print(f"Average Multiplier: {may_data['si_multiplier'].mean():.2f}")
print(f"Multiplier Range: {may_data['si_multiplier'].min():.2f} to {may_data['si_multiplier'].max():.2f}")
print()
print("Strongest May Routes (SI > 1.0):")
strong = may_data[may_data['seasonality_index'] > 1.0][['corridor', 'seasonality_index', 'si_multiplier']].sort_values('seasonality_index', ascending=False)
print(strong.to_string(index=False))
print()
print("Weakest May Routes (SI < 0.9):")
weak = may_data[may_data['seasonality_index'] < 0.9][['corridor', 'seasonality_index', 'si_multiplier']].sort_values('seasonality_index', ascending=False)
print(weak.to_string(index=False))

# Export May data
may_csv = "may_seasonality.csv"
may_summary.to_csv(may_csv, index=False)
print(f"\n[OK] May data exported to: {may_csv}")

MAY SEASONALITY - TARGET CORRIDORS

South Florida -> Chicago            | winter-spring-peak     | SI:  0.01x | Mult:  0.75
South Florida -> New York City      | winter-spring-peak     | SI:  0.01x | Mult:  0.75
Charlotte -> South Florida          | winter-spring-peak     | SI:  0.01x | Mult:  0.75
South Florida -> Denver             | multi-peak-directional | SI:  0.01x | Mult:  0.75
South Florida -> Charlotte          | winter-spring-peak     | SI:  0.01x | Mult:  0.75
New York City -> South Florida      | winter-spring-peak     | SI:  0.01x | Mult:  0.75
Chicago -> South Florida            | winter-spring-peak     | SI:  0.01x | Mult:  0.75
LA Basin -> Denver                  | multi-peak-directional | SI:  0.01x | Mult:  0.75
Denver -> LA Basin                  | multi-peak-directional | SI:  0.01x | Mult:  0.75
Denver -> South Florida             | winter-spring-peak     | SI:  0.01x | Mult:  0.75

MAY SUMMARY STATISTICS
Average SI: 0.01x
SI Range: 0.01x to 0.01x
Average Multiplie

---
# Composite Toggle Score (CTS) - Light Jet Universe
## Formula: `CTS = Direction x TI x SI_Multiplier x LI_Modifier`

### Component Summary
| Component | Input | Output | Grain |
|-----------|-------|--------|-------|
| **SI** | seasonality_index (density x 12) | si_multiplier 0.75-1.50 | corridor x month |
| **TI** | DOW share + TOD share (independent) | ti_score 1.0-10.0 | corridor x DOW x TOD |
| **LI** | WU flights / network WU baseline | li_modifier 0.85-1.30 | corridor x DOW |
| **Direction** | TI + SI gates | +1 / 0 / -1 | corridor x DOW x TOD |
| **CTS** | Direction x TI x SI_Mult x LI_Mod | cts_score float | corridor x month x DOW x TOD |
| **Toggle** | CTS range lookup | toggle_pct % | corridor x month x DOW x TOD |

## 1. Seasonality Intensity (SI) - Multiplier

`seasonality_index = density_pct x 12`  (1.0x = perfectly flat year-round)

Lookup table converts SI to a pricing multiplier:

| SI Range | si_multiplier | Meaning |
|----------|--------------|---------|
| < 0.75x  | 0.75 | Deep off-season |
| 0.75 - 0.90x | 0.90 | Soft demand |
| 0.90 - 1.10x | 1.00 | Near-flat (baseline) |
| 1.10 - 1.30x | 1.15 | Mild seasonal peak |
| 1.30 - 1.60x | 1.30 | Strong seasonal peak |
| > 1.60x  | 1.50 | Extreme seasonal spike |

In [ ]:
# ============================================================
# CTS CELL 1: LOAD lj_comprehensive + SI MULTIPLIER
# ============================================================
comp = comp_out.copy()

MONTH_MAP = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
comp['month_int'] = comp['month'].map(MONTH_MAP)

def si_to_multiplier(si):
    if si < 0.75:   return 0.75
    elif si < 0.90: return 0.90
    elif si < 1.10: return 1.00
    elif si < 1.30: return 1.15
    elif si < 1.60: return 1.30
    else:           return 1.50

comp['si_multiplier'] = comp['seasonality_index'].apply(si_to_multiplier)

print(f"lj_comprehensive: {comp.shape} | corridors: {comp['corridor'].nunique()}")
print("SI Multiplier distribution:")
print(comp['si_multiplier'].value_counts().sort_index())

## 2. Time Intensity (TI) - Decoupled DOW & TOD

DOW and TOD concentration computed **independently** (avoids the sparsity problem of corridor x DOW x TOD joint cells).

**DOW share** = flights on this day / total corridor flights (7 bins per corridor)
**TOD share** = flights in this time window / total corridor flights (6 bins per corridor)
**ti_score** = average(dow_ti, tod_ti)

Percentile rank within corridor -> TI score:

| Percentile (within corridor) | TI Score | Meaning |
|------------------------------|----------|---------|
| <= 10% | 1.0 | Dead slot |
| 10 - 25% | 2.5 | Below average |
| 25 - 50% | 4.5 | Average |
| 50 - 75% | 6.5 | Above average |
| 75 - 90% | 8.5 | High demand slot |
| > 90% | 10.0 | Peak slot |

TOD Buckets:

| Bucket | Time Range | Meaning |
|--------|-----------|---------|
| `00:00-06:59` | Midnight - 6:59am | Overnight/early morning |
| `07:00-09:59` | 7am - 9:59am | Morning departure peak |
| `10:00-12:59` | 10am - 12:59pm | Late morning |
| `13:00-15:59` | 1pm - 3:59pm | Early afternoon |
| `16:00-18:59` | 4pm - 6:59pm | Late afternoon/evening peak |
| `19:00-23:59` | 7pm - 11:59pm | Night |""")

In [ ]:
# ============================================================
# CTS CELL 2: DOW CONCENTRATION - dow_share & dow_ti
# Corridor x DOW only: 360 * 7 = 2,520 rows (dense, no sparsity)
# ============================================================
valid_corridors = comp['corridor'].unique()
lj_cts = lj_lh[lj_lh['corridor'].isin(valid_corridors)].copy()

corridor_totals = (lj_cts.groupby('corridor').size() / 2)

dow_counts = (
    lj_cts.groupby(['corridor', 'dow'])
    .size().div(2)
    .reset_index(name='dow_flight_count')
)
dow_counts['total_corridor_flights'] = dow_counts['corridor'].map(corridor_totals)
dow_counts['dow_share'] = dow_counts['dow_flight_count'] / dow_counts['total_corridor_flights']

dow_counts['dow_ti_pct'] = dow_counts.groupby('corridor')['dow_share'].rank(pct=True) * 100

def percentile_to_ti(p):
    if p <= 10:   return 1.0
    elif p <= 25: return 2.5
    elif p <= 50: return 4.5
    elif p <= 75: return 6.5
    elif p <= 90: return 8.5
    else:         return 10.0

dow_counts['dow_ti'] = dow_counts['dow_ti_pct'].apply(percentile_to_ti)

print(f"DOW concentration rows: {len(dow_counts):,}")
print("DOW share distribution:")
print(dow_counts['dow_share'].describe())
print("\nDOW TI distribution:")
print(dow_counts['dow_ti'].value_counts().sort_index())

In [ ]:
# ============================================================
# CTS CELL 3: TOD CONCENTRATION - tod_share & tod_ti
# Corridor x TOD only: 360 * 8 = 2,880 rows (dense, no sparsity)
# ============================================================

TOD_BINS = [0, 7, 10, 13, 16, 19, 24]
TOD_LABELS = ['00:00-06:59','07:00-09:59','10:00-12:59','13:00-15:59','16:00-18:59','19:00-23:59']

TOD_ORDER  = {lbl: i for i, lbl in enumerate(TOD_LABELS)}

lj_cts['tod_bucket'] = pd.cut(lj_cts['hour'], bins=TOD_BINS, labels=TOD_LABELS, right=False)

tod_counts = (
    lj_cts.groupby(['corridor', 'tod_bucket'])
    .size().div(2)
    .reset_index(name='tod_flight_count')
)
tod_counts['total_corridor_flights'] = tod_counts['corridor'].map(corridor_totals)
tod_counts['tod_share'] = tod_counts['tod_flight_count'] / tod_counts['total_corridor_flights']
tod_counts['tod_order'] = tod_counts['tod_bucket'].map(TOD_ORDER)

tod_counts['tod_ti_pct'] = tod_counts.groupby('corridor')['tod_share'].rank(pct=True) * 100
tod_counts['tod_ti'] = tod_counts['tod_ti_pct'].apply(percentile_to_ti)

print(f"TOD concentration rows: {len(tod_counts):,}")
print("TOD share distribution:")
print(tod_counts['tod_share'].describe())
print("\nTOD TI distribution:")
print(tod_counts['tod_ti'].value_counts().sort_index())

In [ ]:
# ============================================================
# CTS CELL 4: COMBINED TI SCORE - cross-join DOW x TOD
# Average independent DOW_TI and TOD_TI scores
# ============================================================
ti_combined = dow_counts[['corridor','dow','dow_flight_count','dow_share','dow_ti_pct','dow_ti']].merge(
    tod_counts[['corridor','tod_bucket','tod_order','tod_share','tod_ti_pct','tod_ti']],
    on='corridor', how='inner'
)
ti_combined['ti_score'] = ((ti_combined['dow_ti'] + ti_combined['tod_ti']) / 2).round(2)

print(f"CTS universe shape: {ti_combined.shape}")
print(f"Corridors: {ti_combined['corridor'].nunique()}")
print(f"Unique dow: {ti_combined['dow'].nunique()}")
print(f"Unique tod_bucket: {ti_combined['tod_bucket'].nunique()}")
print("\nTI Score distribution:")
print(ti_combined['ti_score'].value_counts().sort_index())

## 3. Loyalty Intensity (LI) - Wheels Up Capture Strength

`li_index = dow_wu_share / network_wu_share`

Computed at **corridor x DOW** level (dense). Compares WU's share on a specific day against WU's average share across the entire LJ network.

- li_index = 1.0 means WU performs exactly at network average on this corridor/day
- li_index = 2.5 means WU captures 2.5x more share than network average

| LI Index | li_modifier | Meaning |
|----------|------------|---------|
| < 0.75x  | 0.85 | WU under-indexed - weak position |
| 0.75 - 1.00x | 0.95 | Slightly below average |
| 1.00 - 1.50x | 1.00 | Near network baseline |
| 1.50 - 2.50x | 1.10 | WU strong on this corridor/day |
| 2.50 - 4.00x | 1.20 | WU dominant |
| > 4.00x  | 1.30 | WU highly concentrated here |

In [ ]:
# ============================================================
# CTS CELL 5: LOYALTY INTENSITY - WU share at corridor x DOW level
# Stays at DOW grain (dense 360x7) to avoid sparsity when adding TOD
# li_index = dow_wu_share / network_wu_share
# ============================================================
network_wu_share = lj_cts['is_wup'].sum() / len(lj_cts)

wu_dow = (
    lj_cts[lj_cts['is_wup']]
    .groupby(['corridor', 'dow'])
    .size().div(2)
    .reset_index(name='dow_wu_flights')
)

# Merge WU at corridor x DOW level into ti_combined
cell_li = ti_combined.merge(wu_dow, on=['corridor', 'dow'], how='left')
cell_li['dow_wu_flights'] = cell_li['dow_wu_flights'].fillna(0)
cell_li['dow_wu_share'] = cell_li['dow_wu_flights'] / cell_li['dow_flight_count'].replace(0, np.nan)
cell_li['network_wu_share'] = network_wu_share
cell_li['li_index'] = (cell_li['dow_wu_share'] / network_wu_share).fillna(0)

print(f"Network WU share: {network_wu_share:.2%}")
print("LI Index distribution:")
print(cell_li['li_index'].describe())

In [ ]:
# ============================================================
# CTS CELL 6: LI MODIFIER - step function on li_index
# ============================================================
def li_to_modifier(li):
    if li < 0.75:   return 0.85
    elif li < 1.00: return 0.95
    elif li < 1.50: return 1.00
    elif li < 2.50: return 1.10
    elif li < 4.00: return 1.20
    else:           return 1.30

cell_li['li_modifier'] = cell_li['li_index'].apply(li_to_modifier)

print("LI Modifier distribution:")
print(cell_li['li_modifier'].value_counts().sort_index())

## 4. Direction - TI & SI Gate-keeper

TI is the primary gate. SI alone cannot create a positive toggle.

| TI Score | SI Multiplier | Direction | Rationale |
|----------|--------------|-----------|-----------|
| >= 7 | any | **+1** | Peak demand slot - always positive |
| 4-7 | >= 1.0 | **+1** | Moderate demand AND in-season |
| 4-7 | < 1.0 | **0** | Moderate demand but off-season |
| < 4 | < 0.90 | **-1** | Weak demand AND off-season |
| < 4 | >= 0.90 | **0** | Weak demand but near-flat season |

## 5. CTS Score & Toggle Mapping

`CTS = Direction x TI x SI_Multiplier x LI_Modifier`

When Direction = 0, CTS = 0 regardless of other factors.
When Direction = -1, CTS is negative (counter-cyclical discount signal).

| CTS Score | toggle_pct | Action |
|-----------|-----------|--------|
| <= -6 | -15% | Deep counter-cyclical discount |
| -6 to -3 | -10% | Moderate discount |
| -3 to +3 | 0% | Hold base price |
| +3 to +6 | +5% | Mild premium |
| +6 to +9 | +10% | Moderate premium |
| +9 to +12 | +15% | Strong premium |
| +12 to +15 | +20% | High-demand premium |
| >= +15 | +25% | Peak pricing |

In [ ]:
# ============================================================
# CTS CELL 7: MERGE - add SI by corridor x month
# ============================================================
si_base = comp[['corridor','month','month_int','cluster_label',
                'density_pct','seasonality_index','slab','intensity','si_multiplier']].copy()

cts = cell_li.merge(si_base, on='corridor', how='inner')

print(f"CTS universe shape: {cts.shape}")
print(f"Unique corridor-month: {cts.groupby(['corridor','month']).ngroups}")

In [ ]:
# ============================================================
# CTS CELL 8: DIRECTION - TI and SI gate-keeper rules
# ============================================================
def get_direction(ti, si_mult):
    if ti >= 7:                        return  1
    elif ti >= 4 and si_mult >= 1.0:   return  1
    elif ti >= 4:                      return  0
    elif si_mult < 0.90:               return -1
    else:                              return  0

cts['direction'] = cts.apply(
    lambda r: get_direction(r['ti_score'], r['si_multiplier']), axis=1
)
print("Direction distribution:")
print(cts['direction'].value_counts())

In [ ]:
# ============================================================
# CTS CELL 9: CTS SCORE = Direction * TI * SI_Mult * LI_Mod
# ============================================================
cts['cts_score'] = (
    cts['direction'] * cts['ti_score'] *
    cts['si_multiplier'] * cts['li_modifier']
).round(2)

print("CTS Score summary:")
print(cts['cts_score'].describe())
print(f"\nCTS > 0: {(cts['cts_score'] > 0).sum():,}")
print(f"CTS = 0: {(cts['cts_score'] == 0).sum():,}")
print(f"CTS < 0: {(cts['cts_score'] < 0).sum():,}")

In [ ]:
# ============================================================
# CTS CELL 10: TOGGLE RECOMMENDATION - CTS -> toggle_pct
# ============================================================
def cts_to_toggle(v):
    if v <= -6:   return -15
    elif v <= -3: return -10
    elif v < 3:   return   0
    elif v < 6:   return   5
    elif v < 9:   return  10
    elif v < 12:  return  15
    elif v < 15:  return  20
    else:         return  25

cts['toggle_pct'] = cts['cts_score'].apply(cts_to_toggle)
print("Toggle distribution:")
print(cts['toggle_pct'].value_counts().sort_index())

In [ ]:
# ============================================================
# CTS CELL 11: EXPORT - lj_cts_universe.csv
# ============================================================
OUT_CTS_PATH = "lj_cts_universe.csv"

EXPORT_COLS = [
    'cluster_label','corridor','month','month_int',
    'density_pct','seasonality_index','slab','intensity',
    'dow','dow_share','dow_ti','dow_ti_pct',
    'tod_bucket','tod_order','tod_share','tod_ti','tod_ti_pct',
    'ti_score',
    'si_multiplier',
    'dow_wu_flights','dow_wu_share','network_wu_share','li_index','li_modifier',
    'direction','cts_score','toggle_pct',
]

cts_out = cts[EXPORT_COLS].sort_values(
    ['cluster_label','corridor','month_int','dow','tod_order']
).reset_index(drop=True)

cts_out.to_csv(OUT_CTS_PATH, index=False)
print(f"Exported: {OUT_CTS_PATH}")
print(f"Shape: {cts_out.shape}")
cts_out.head(5)

In [ ]:
# ============================================================
# FINAL VALIDATION: LJ UNIVERSE & VOLUME DISTRIBUTION
# ============================================================
sep = "=" * 60

print(sep); print("UNIVERSE SUMMARY"); print(sep)
print(f"  Corridors                 : {cts_out['corridor'].nunique()}")
print(f"  Total rows                : {len(cts_out):,}")
print(f"  Months                    : 12 (Jan-Dec)")
print(f"  Days of week per corridor : 7")
print(f"  TOD buckets per corridor  : 8")
print()

print(sep); print("DATA DENSITY CHECK"); print(sep)
print(f"  Expected max rows (360 * 12 * 7 * 8) : {360*12*7*8:,}")
print(f"  Actual rows                           : {len(cts_out):,}")
print(f"  Occupancy                             : {len(cts_out)/(360*12*7*8)*100:.1f}%")
print(f"  Non-zero dow_share                    : {(cts_out['dow_share'] > 0).sum():,}")
print(f"  Non-zero tod_share                    : {(cts_out['tod_share'] > 0).sum():,}")
print()

print(sep); print("TOGGLE DISTRIBUTION"); print(sep)
tgl = cts_out['toggle_pct'].value_counts().sort_index()
for pct, cnt in tgl.items():
    bar = "█" * max(1, cnt // max(tgl.max() // 40, 1))
    print(f"  {pct:+4d}%  {cnt:6,}  {bar}")
print()

print(sep); print("CTS SCORE STATISTICS"); print(sep)
print(cts_out['cts_score'].describe().round(2).to_string())
print()

print(sep); print("TOP 10 POSITIVE CTS CORRIDORS"); print(sep)
top_pos = (cts_out[cts_out['cts_score'] > 0]
           .nlargest(10, 'cts_score')
           [['corridor','month','dow','tod_bucket','ti_score',
             'si_multiplier','li_modifier','cts_score','toggle_pct']])
print(top_pos.to_string(index=False))
print()

print(sep); print("CLUSTER BREAKDOWN (avg CTS by archetype)"); print(sep)
clust = (cts_out.groupby('cluster_label')['cts_score']
         .agg(['mean','min','max','count']).round(2))
print(clust.to_string())